## Neural Network Implementations Development 
This purpose this notebook is to incrementally implement a neural network to strengthen understanding of the algorithm

In [1]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd 
import seaborn as sns 

from typing import Tuple

## Forward Pass for 1 Hidden Layer
Through making use of matrix operation of weights, bias and activation through numpy the model is able to make prediction. Also For simplicity in batch operations, I chose row-vector inputs (samples × features), which is equivalent to the column-vector convention seen in textbooks.

In [11]:
def enforce_row_vector(x: np.ndarray) -> np.ndarray:
    """Force input into shape (1, n_features) if the input is 1 dimensional."""
    x = np.array(x, dtype=float)        # ensure NumPy array
    if x.ndim == 1: 
        return x.reshape(1, -1 ) # row vector
    else:
        return x 

In [2]:
def forward_pass_1HL(feature_array: np.ndarray,hidden_weights: np.ndarray, 
                     hidden_bias:np.ndarray, out_weights: np.ndarray, out_bias:np.ndarray,
                    target_vector: np.ndarray)-> Tuple[np.ndarray, np.ndarray, float, np.ndarray]:
    '''
        Conducts the forward pass of a 1 hidden layer neural net using the ReLu activation function
        and evaluates the results using MSE
        
        Args:
            feature_array (np.ndarray): Inputted feature array where each col represents a feature and each row a sample 
            hidden_weights (np.ndarray): Weight matrix for the hidden layer
            hidden_bias (np.ndarray): Bias vector for the hidden layer
            out_weights (np.ndarray) Weight matrix for the output layer
            out_bias (np.ndarray): Bias vector for the output layer
            target_vector (np.ndarray): Target vector for preformance evaluation. Where each col represents a target and each row a sample.   
        Return:
            np.ndarray: Output activations of the network
            np.ndarray: The difference vector of the output activations and the true y
            float: The mean squared error of the network's predictions
            np.ndarray: The neuron activation values of the last hidden layer
    '''
    feature_array = enforce_row_vector(feature_array)
    
    hidden_activations = np.maximum(feature_array @ hidden_weights + hidden_bias, 0)
    output_activations = hidden_activations @ out_weights + out_bias
    error = (target_vector - output_activations)
    half_mse = (1/2) * np.mean((error) ** 2)
    return output_activations, error, half_mse, hidden_activations 
    

## Computing Gradients 
Through using a linear algebra implementions of weights and biases gradient formulas obtained from back propogation we can get the gradient vectors of each layer. This was derived through assessing that the gradient/effect of a weight is dependant on the initial activation it is associated with, the activations it directly contributes to and the future activations down the line. Linear algebra can capture this realtionship and since the equations are implemented through matrix algebra we can use numpy vector operations for efficiency, avoiding the the cost of using loops.   

In [3]:
def compute_output_gradients(error: np.ndarray, hidden_activations:np.ndarray) -> np.ndarray:
    '''
    Computes the gradients of the weights and biases in the output layer

    Args: 
        error (np.ndarray): The difference vector of the output activations and the true y 
        hidden_activations (np.ndarray): The neuron activation values of the last hidden layer

    Returns:
        np.ndarray: Gradient array of the output units containing the gradient values of the biases
        np.ndarray: Gradient matrix of the output units containing the gradient values of the weights
        
    '''
 
    bias_gradients = np.ones((1, hidden_activations.shape[1])) @ (-1 * error ).T
    weight_gradients = hidden_activations.T @ (-1 * error)
    return bias_gradients, weight_gradients 

In [4]:
def compute_last_hidden_gradients(error: np.ndarray, hidden_activations:np.ndarray, 
                             out_weights: np.ndarray, feature_array: np.ndarray) -> np.ndarray:
    '''
    Computes the gradients of the weights and biases in the hidden layer.

    Args: 
        feature_array (np.ndarray): Inputted feature array
        error (np.ndarray): The difference vector of the output activations and the true y 
        hidden_activations (np.ndarray): The neuron activation values of the last hidden layer
        out_weights (np.ndarray): The weight matrix for the output neurons 

    Returns:
        np.ndarray: Gradient array of the hidden units containing the gradient values of the biases
        np.ndarray: Gradient matrix of the hidden units containing the gradient values of the weights
        
    
    '''
    relu_mask = hidden_activations > 0
    relu_derivative = relu_mask.astype(int)
    bias_gradients = np.ones(( feature_array.shape[1], 1)) @ ( -1 * error @ out_weights.T ) * relu_derivative
    weight_gradients = feature_array.T @ (-1 * error @ out_weights.T ) * relu_derivative
    return bias_gradients, weight_gradients

## Back Propagation 
Since the forward pass function allows for batch inputs, the loss of the whole network can be calculated by inputting the full training input into the forward pass function. When the outputs of the forward pass is inputted into the gradient functions the resulting output is essentially a summation of the gradient values of the weights and biases for each training example. Thus to get the mean gradients values you simply have to divide the arrays by 1/n, where n is the sample size.   

In [ ]:
def back_propagation(training_feature_array: np.ndarray,hidden_weights: np.ndarray, 
                     hidden_bias:np.ndarray, out_weights: np.ndarray, out_bias:np.ndarray,
                    target_matrix: np.ndarray)-> Tuple[np.ndarray, np.ndarray]:
    '''
    Runs  backpropogation on the training data to generate the mean gradients of the weights and biases for 
    a 1 hidden layer neural network.

    Args:
        training_feature_array (np.ndarray): Inputted training feature array.  Each row represent xi
        hidden_weights (np.ndarray): Weight matrix for the hidden layer
        hidden_bias (np.ndarray): Bias vector for the hidden layer
        out_weights (np.ndarray) Weight matrix for the output layer
        out_bias (np.ndarray): Bias vector for the output layer
        target_matrix (np.ndarray): Target matrix for preformance evaluation. Each row represent yi
    Return:
        np.ndarray: The mean output layer bias gradients 
        np.ndarray: The mean output layer weight gradients 
        np.ndarray: The mean hidden layer bias gradients 
        np.ndarray: The mean hidden layer weight gradients 
    '''
 
    
    output_activations, error, half_mse, hidden_activations = forward_pass_1HL(training_feature_array, hidden_weights, 
                         hidden_bias, out_weights, out_bias,
                        target_matrix)
    
    output_bias_gradients, output_weight_gradients = compute_output_gradients(error, hidden_activations, 
                             out_weights, training_feature_array)
        
    hidden_bias_gradients, hidden_weight_gradients = compute_last_hidden_gradients(error, hidden_activations, 
                                 out_weights, training_feature_array)

    n = len(training_feature_array)
    mean_out_weight_gradients = (1/n) * output_weight_gradients
    mean_out_bias_gradients = (1/n) * output_bias_gradients
    mean_hidden_weight_gradients = (1/n) * hidden_weight_gradients
    mean_hidden_bias_gradients = (1/n) * hidden_bias_gradients

    return mean_out_bias_gradients, mean_out_weight_gradients, mean_hidden_bias_gradients, mean_hidden_weight_gradients
    

## Gradient Updating
Once we have the gradient vectors and matrices we can update the weight and biase values through the regular convention using the chosen learning rate. 

In [6]:
def gradient_descent_update(learning_rate: float, old_out_weights:np.ndarray, 
                            old_out_bias: np.ndarray, output_bias_gradients: np.ndarray, 
                            output_weight_gradients: np.ndarray, old_hidden_weights: np.ndarray, 
                            old_hidden_bias: np.ndarray, hidden_bias_gradients: np.ndarray, 
                            hidden_weight_gradients: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Conducts gradient descent updating of the weights and biases of a 1 hidden layer neural network

    Args: 
        learning_rate (float): The learning rate for the neural network 
        old_out_weights (np.ndarray): The current weights of the output layer  
        old_out_bias(np.ndarray): The current biases of the output layer
        output_bias_gradients (np.ndarray): The gradient values of the current output layer biases
        output_weights_gradients (np.ndarray): The gradient values of the current output layer weights
        old_hidden_weights(np.ndarray): The current weights of the hidden layer 
        old_hidden_bias (np.ndarray): The current biases of the hidden layer 
        hidden_bias_gradients (np.ndarray):  The gradient values of the current hidden layer biases
        hidden_bias_gradients (np.ndarray): The gradient values of the current hidden layer weights
    
    Return:
        np.ndarray: Updated output layer weights
        np.ndarray: Updated output layer biases
        np.ndarray: Updated hidden layer weights
        np.ndarray: Updated hidden layer biases

    '''

    updated_output_weights = old_out_weights - learning_rate * output_weight_gradients
    updated_output_biases = old_out_bias - learning_rate * output_bias_gradients
    updated_hidden_weights = old_hidden_weights - learning_rate * hidden_weight_gradients
    updated_hidden_biases = old_hidden_bias - learning_rate * hidden_weight_gradients

    return updated_output_weights, updated_output_biases, updated_hidden_weights, updated_hidden_biases

## 1 Hidden Layer Nerual Net with early stopping validation and HE initialisation 
To deal with weight initailisation, I went with He as it would work well with the ReLu activation function used through the network. In future iterations, other initialisation options will be added along with other activation functions. In regards to the stopping mechanisms, “I used early stopping on validation loss with a patience of k epochs. This prevents overfitting while also providing a clear stopping mechanism. This I felt was the most practical option, however, mechanisms like gradient norm threshold do deserve experimentation even if they are more unstable, so they will be added in the next stage as an option when the nerual net is wrapped as a class.

In [ ]:
def neural_net_1HL(training_feature_array: np.ndarray, hidden_neurons: int, 
                    target_matrix: np.ndarray, learning_rate: float, 
                   max_epochs: int, validation_features: np.ndarray, 
                   validations_target: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Conducts gradient descent on a 1 hidden layer neural network to output the trained weights for predictions

    Args:
        training_feature_array (np.ndarray): Inputted training feature array.  Each row represent xi
        hidden_neurons (int): Number of neuron in the hidden layer
        target_matrix (np.ndarray): Target matrix for preformance evaluation. Each row represent yi
        learning_rate (float): The learning rate for the neural network 
        max_epochs (int): Max number of iterations of gradient descent to conduct.
        validation_features (np.ndarray): Validation feature array to assess improvements in the neural net for stopping criteria. 
        validation_features (np.ndarray): Validation target array to assess improvements in the neural net for stopping criteria. 
        k (int): Number of epochs allowed without a change in validation loss before stopping gradient descent  
    Returns:
        np.ndarray: Trained output layer weights
        np.ndarray: Trained output layer biases
        np.ndarray: Trained hidden layer weights
        np.ndarray: Trained hidden layer biases
    '''
    hidden_weights = np.random.rand(training_feature_array.shape[1], hidden_neurons) * np.sqrt(2 / training_feature_array.shape[1])
    hidden_bias = np.zeros((1, hidden_neurons))
    out_weights =  np.random.rand(hidden_neurons, target_matrix.shape[1]) * np.sqrt(2 / hidden_neurons)
    out_bias = np.zeros((1, target_matrix.shape[1]))
    previous_validation_loss = 0
    epoch_completed = 0
    no_change_epochs = 0 
    epoch_count = 0 
    while not stop_descent:
        (out_bias_gradients, out_weight_gradients, 
         hidden_bias_gradients, hidden_weight_gradients) = back_propagation(training_feature_array ,
                                                                            hidden_weights, hidden_bias, 
                                                                            out_weights, out_bias,
                                                                                    target_matrix)
        
        output_weights, output_biases, hidden_weights, hidden_biases= gradient_descent_update(learning_rate, out_weights, 
                                                                                                out_bias, out_bias_gradients, 
                                                                                                out_weight_gradients, hidden_weights, 
                                                                                                hidden_bias, hidden_bias_gradients, 
                                                                                                hidden_weight_gradients)
        validation_loss = forward_pass_1HL(validation_features, hidden_weights, 
                     hidden_biases, out_weights, out_biases,
                    validation_target)[2]
        if validation_loss == previous_validation_loss:
            no_change_epochs += 1 
        else:
            no_change_epochs = 0
        
        previous_validation_loss = validation_loss

        if no_change_epochs >= k:
            stop_descent = True

        epoch_count += 1 
        if epoch_count >= max_epochs:
            stop_descent = True 
            

        
    return output_weights, output_biases, hidden_weights, hidden_biases 
        

## 2 Hidden Layer Neural Net 
For the purposes of investigating multi hiddel layer networks, a two hidden layer function will be created. I implemented the 1-hidden-layer network using the row-vector convention, which is consistent with how most ML libraries treat datasets (n_samples × n_features). For the 2-hidden-layer version, I considered switching to the column-vector convention (common in textbooks), but decided to keep it consistent for implementation clarity.

## 2 Layer Forward Pass
The fundamental function does change much when we add a second layer. Of course now with another layer, there needs to be new inputs and a new output to account for the layer however, in the function it self the additional layer operation can be account for in just one extra line of code and some slight alterations. Got to love Numpy and linear algebra 

In [ ]:
def enforce_row_vector(x: np.ndarray) -> np.ndarray:
    """Force input into shape (1, n_features) if the input is 1 dimensional."""
    x = np.array(x, dtype=float)        # ensure NumPy array
    if x.ndim == 1: 
        return x.reshape(1, -1 ) # row vector
    else:
        return x 

In [12]:
def forward_pass_2HL(feature_array: np.ndarray, hidden1_weights: np.ndarray, hidden1_bias: np.ndarray ,hidden2_weights: np.ndarray, 
                     hidden2_bias:np.ndarray, out_weights: np.ndarray, out_bias:np.ndarray,
                    target_matrix: np.ndarray)-> Tuple[np.ndarray, np.ndarray, float, np.ndarray]:
    '''
        Conducts the forward pass of a 2 hidden layer neural net using the ReLu activation function
        and evaluates the results using MSE
        
        Args:
            feature_array (np.ndarray): Inputted feature array where each col represents a feature and each row a sample 
            hidden1_weights (np.ndarray): Weight matrix for the first hidden layer
            hidden1_bias (np.ndarray): Bias vector for the first hidden layer
            hidden2_weights (np.ndarray): Weight matrix for the second hidden layer
            hidden2_bias (np.ndarray): Bias vector for the second hidden layer
            out_weights (np.ndarray) Weight matrix for the output layer
            out_bias (np.ndarray): Bias vector for the output layer
            target_matrix (np.ndarray): Target matrix for preformance evaluation. Where each col represents a target and each row a sample.   
        Return:
            np.ndarray: Output activations of the network
            np.ndarray: The difference vector of the output activations and the true y
            float: The mean squared error of the network's predictions
            np.ndarray: The neuron activation values of the first hidden layer 
            np.ndarray: The neuron activation values of the last hidden layer
    '''
    feature_array = enforce_row_vector(feature_array)
    
    hidden1_activations = np.maximum(feature_array @ hidden1_weights + hidden1_bias, 0)
    hidden2_activations = np.maximum(hidden1_activations @ hidden2_activations + hidden2_bias, 0)
    output_activations = hidden2_activations @ out_weights + out_bias
    error = (target_matrix - output_activations)
    half_mse = (1/2) * np.mean((error) ** 2)
    return output_activations, error, half_mse, hidden1_activations, hidden2_activations 
    

## Computing Gradients with 2 Hidden Layers 
With a 2 layer network, the first two functions do not change much, most material change being which neuron activation affect the specific weight being assessed. In the case of the first hidden layer we see that that matrix euation follows the similar pattern where the gradient is determined by the activation the weight affects and then its contribution to the error through the subsiquent connections. 

In [ ]:
def compute_output_gradients(error: np.ndarray, hidden2_activations:np.ndarray) ->  Tuple[np.ndarray, np.ndarray]:
    '''
    Computes the gradients of the weights and biases in the output layer

    Args: 
        error (np.ndarray): The difference vector of the output activations and the true y 
        hidden2_activations (np.ndarray): The neuron activation values of the last hidden layer

    Returns:
        np.ndarray: Gradient array of the output units containing the gradient values of the biases
        np.ndarray: Gradient matrix of the output units containing the gradient values of the weights
        
    '''
 
    bias_gradients = np.ones(( hidden2_activations[1].shape[1], 1)) @ (-1 * error )
    weight_gradients = hidden2_activations.T @ (-1 * error)
    return bias_gradients, weight_gradients 

In [ ]:
def compute_last_hidden_gradients(error: np.ndarray, out_weights: np.ndarray, 
                                  hidden2_activations: np.ndarray, hidden1_activations: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    '''
    Computes the gradients of the weights and biases in the last hidden layer.

    Args: 
        hidden1_activations (np.ndarray): The neuron activation values of the first hidden layer
        error (np.ndarray): The difference vector of the output activations and the true y 
        hidden2_activations (np.ndarray): The neuron activation values of the last hidden layer
        out_weights (np.ndarray): The weight matrix for the output neurons 

    Returns:
        np.ndarray: Gradient array of the hidden units containing the gradient values of the biases
        np.ndarray: Gradient matrix of the hidden units containing the gradient values of the weights
    '''
    relu_mask = hidden2_activations > 0
    relu_derivative = relu_mask.astype(int)
    bias_gradients = np.ones((hidden1_activations.shape[1], 1 )) @ ( -1 * error @ out_weights.T ) * relu_derivative
    weight_gradients = hidden1_activations.T @ (-1 * error @ out_weights.T ) * relu_derivative
    return bias_gradients, weight_gradients

In [ ]:
def compute_first_hidden_gradients(feature_array: np.ndarray ,error: np.ndarray, hidden2_weights: np.ndarray, 
                             out_weights: np.ndarray, hidden1_activations: np.ndarray ) -> Tuple[np.ndarray, np.ndarray]:
    '''
    Computes the gradients of the weights and biases in the first hidden layer.

    Args: 
        feature_array (np.ndarray): Inputted feature array where each col represents a feature and each row a sample 
        hidden1_activations (np.ndarray): The neuron activation values of the first hidden layer
        error (np.ndarray): The difference vector of the output activations and the true y 
        hidden2_weights (np.ndarray): The  weight values of the last hidden layer
        out_weights (np.ndarray): The weight matrix for the output neurons 

    Returns:
        np.ndarray: Gradient array of the hidden units containing the gradient values of the biases
        np.ndarray: Gradient matrix of the hidden units containing the gradient values of the weights
    '''
    relu_mask = hidden1_activations > 0
    relu_derivative = relu_mask.astype(int)
    bias_gradients = np.ones( feature_array.shape[1], 1) @ (-1 error @ out_weights.T) @ out_weights.T * relu_derivative
    weight_gradients = feature_array.T  @ (-1 error @ out_weights.T) @ out_weights.T * relu_derivative
    return bias_gradients , weight_gradients

## Back Propagation with 2 Hidden Layers 
With additional layer, back propagation must go back one extra layer to obtain the gradients for all the weights and biases for the network. Thus another function to calculate and return these gradients is used. Also the specific inputs for the other gradient calculations change as a result of the change in there relative positions in the network compared to a 1 Layer network. Still the function is still fundamentally the same, and while adding gradient calculation functions is not exactly the most scalable method, it is still clear that not many changes need to be made accomodate networks with deeper architectures. In future class based implementation though, a more general and adaptable method will be investigated.

In [ ]:
def back_propagation(feature_array: np.ndarray, hidden1_weights: np.ndarray, hidden1_bias: np.ndarray ,   
                    hidden2_weights: np.ndarray, 
                     hidden2_bias:np.ndarray, out_weights: np.ndarray, out_bias:np.ndarray,
                    target_matrix: np.ndarray)-> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Runs  backpropogation on the training data to generate the mean gradients of the weights and biases for 
    a 1 hidden layer neural network.

    Args:
        feature_array (np.ndarray): Inputted feature array where each col represents a feature and each row a sample 
        hidden1_weights (np.ndarray): Weight matrix for the first hidden layer
        hidden1_bias (np.ndarray): Bias vector for the first hidden layer
        hidden2_weights (np.ndarray): Weight matrix for the second hidden layer
        hidden2_bias (np.ndarray): Bias vector for the second hidden layer
        out_weights (np.ndarray) Weight matrix for the output layer
        out_bias (np.ndarray): Bias vector for the output layer
        target_matrix (np.ndarray): Target matrix for preformance evaluation. Where each col represents a target and each row a sample.   
    Return:
        np.ndarray: The mean output layer bias gradients 
        np.ndarray: The mean output layer weight gradients 
        np.ndarray: The mean last hidden layer bias gradients 
        np.ndarray: The mean last hidden layer weight gradients
        np.ndarray: The mean first hidden layer bias gradients
        np.ndarray: The mean first hidden layer weight gradients 
    '''
 
    
    (_, error, _, hidden1_activations, 
    hidden2_activations) = forward_pass_2HL(feature_array, hidden1_weights, 
                                            hidden1_bias, hidden2_weights, 
                                            hidden2_bias, out_weights, out_bias,
                                            target_matrix)
    
    output_bias_gradients, output_weight_gradients = compute_output_gradients(error, hidden2_activations)
        
    hidden2_bias_gradients, hidden2_weight_gradients = compute_last_hidden_gradients(error, out_weights, 
                                                                            hidden2_activations, hidden1_activations)
    
    (hidden1_bias_gradients, 
     hidden1_weight_gradients) = compute_first_hidden_gradients(feature_array, error,      hidden2_weights,                                                      
            out_weights, hidden1_activations)

    n = len(feature_array)
    mean_out_weight_gradients = (1/n) * output_weight_gradients
    mean_out_bias_gradients = (1/n) * output_bias_gradients
    mean_hidden2_weight_gradients = (1/n) * hidden2_weight_gradients
    mean_hidden2_bias_gradients = (1/n) * hidden2_bias_gradients
    mean_hidden1_weight_gradients = (1/n) * hidden1_weight_gradients
    mean_hidden1_bias_gradients = (1/n) * hidden1_bias_gradients
    return (mean_out_bias_gradients, mean_out_weight_gradients, mean_hidden2_bias_gradients,      mean_hidden2_weight_gradients, mean_hidden1_bias_gradients, mean_hidden1_weight_gradients)
    

## Gradient Updating  with 2 Hidden Layers 
The additional layer requires simple additions to the function to update and output the gradients of extra layer. There are 4 additional inputs and two addtional outputs to account for the extra layer.  

In [ ]:
def gradient_descent_update(learning_rate: float, old_out_weights:np.ndarray, 
                            old_out_bias: np.ndarray, output_bias_gradients: np.ndarray, 
                            output_weight_gradients: np.ndarray, old_hidden2_weights: np.ndarray, 
                            old_hidden2_bias: np.ndarray, hidden2_bias_gradients: np.ndarray, 
                            hidden2_weight_gradients: np.ndarray, old_hidden1_weights: np.ndarray, 
                            old_hidden1_bias: np.ndarray, hidden1_bias_gradients: np.ndarray, 
                            hidden1_weight_gradients: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Conducts gradient descent updating of the weights and biases of a 2 hidden layer neural network

    Args: 
        learning_rate (float): The learning rate for the neural network 
        old_out_weights (np.ndarray): The current weights of the output layer  
        old_out_bias(np.ndarray): The current biases of the output layer
        output_bias_gradients (np.ndarray): The gradient values of the current output layer biases
        output_weights_gradients (np.ndarray): The gradient values of the current output layer weights
        old_hidden2_weights(np.ndarray): The current weights of the last hidden layer 
        old_hidden2_bias (np.ndarray): The current biases of the last hidden layer 
        hidden2_bias_gradients (np.ndarray):  The gradient values of the current final hidden layer biases
        hidden2_bias_gradients (np.ndarray): The gradient values of the current final  hidden layer weights
        old_hidden1_weights(np.ndarray): The current weights of the first hidden layer 
        old_hidden1_bias (np.ndarray): The current biases of the first hidden layer 
        hidden1_bias_gradients (np.ndarray):  The gradient values of the current first hidden layer biases
        hidden1_bias_gradients (np.ndarray): The gradient values of the current first  hidden layer weights
    
    
    Return:
        np.ndarray: Updated output layer weights
        np.ndarray: Updated output layer biases
        np.ndarray: Updated last hidden layer weights
        np.ndarray: Updated last hidden layer biases
        np.ndarray: Updated first hidden layer weights
        np.ndarray: Updated first hidden layer biases

    '''

    updated_output_weights = old_out_weights - learning_rate * output_weight_gradients
    updated_output_biases = old_out_bias - learning_rate * output_bias_gradients
    updated_hidden2_weights = old_hidden2_weights - learning_rate * hidden2_weight_gradients
    updated_hidden2_biases = old_hidden2_bias - learning_rate * hidden2_bias_gradients
    updated_hidden1_weights = old_hidden1_weights - learning_rate * hidden1_weight_gradients
    updated_hidden1_biases = old_hidden1_bias - learning_rate * hidden1_bias_gradients

    return (updated_output_weights, updated_output_biases, updated_hidden2_weights, updated_hidden2_biases, 
            updated_hidden1_weights, updated_hidden1_biases)

## Neural Net with 2 Hidden Layers 
With the changes in the other functions the changes in the final function, are mainly in the vain of expanding the initialization protocol to account  for the new layer, and expanding the output to return the extra layer weights. Only one extra input is need (hidden1_neurons) which allows of the initalization of all the required weights and biases. 

In [ ]:
def neural_net_2HL(training_feature_array: np.ndarray, hidden1_neurons: int, hidden2_neurons:int,
                    target_matrix: np.ndarray, learning_rate: float, 
                   max_epochs: int, validation_features: np.ndarray, 
                   validation_target: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Conducts gradient descent on a 2 hidden layer neural network to output the trained weights for predictions. Uses He Initialization and 
    Validation loss with max epochs stopping criteria. 

    Args:
        training_feature_array (np.ndarray): Inputted training feature array.  Each row represent xi
        hidden1_neurons (int): Number of neuron in the first hidden layer
        hidden2_neurons (int): Number of neuron in the second hidden layer
        target_matrix (np.ndarray): Target matrix for preformance evaluation. Each row represent yi
        learning_rate (float): The learning rate for the neural network 
        max_epochs (int): Max number of iterations of gradient descent to conduct.
        validation_features (np.ndarray): Validation feature array to assess improvements in the neural net for stopping criteria. 
        validation_features (np.ndarray): Validation target array to assess improvements in the neural net for stopping criteria. 
        k (int): Number of epochs allowed without a change in validation loss before stopping gradient descent  
    Returns:
        np.ndarray: Trained output layer weights
        np.ndarray: Trained output layer biases
        np.ndarray: Trained last hidden layer weights
        np.ndarray: Trained last hidden layer biases
        np.ndarray: Trained first hidden layer weights
        np.ndarray: Trained first hidden layer biases
    '''
    hidden1_weights = np.random.rand(training_feature_array.shape[1], hidden1_neurons) * np.sqrt(2 / training_feature_array.shape[1])
    hidden1_bias = np.zeros((1, hidden1_neurons))
    hidden2_weights = np.random.rand(hidden1_neurons, hidden2_neurons) * np.sqrt(2 / hidden1_neurons)
    hidden2_bias = np.zeros((1, hidden2_neurons))
    out_weights =  np.random.rand(hidden2_neurons, target_matrix.shape[1]) * np.sqrt(2 / hidden2_neurons)
    out_bias = np.zeros((1, target_matrix.shape[1]))
    
    previous_validation_loss = 0
    epoch_completed = 0
    no_change_epochs = 0  # Number of epochs without an improvement in validation loss 
    epoch_count = 0 
    while not stop_descent:
        (out_bias_gradients, out_weight_gradients, hidden2_bias_gradients, 
         hidden2_weight_gradients, hidden1_bias_gradients, 
         hidden1_weight_gradients) = back_propagation(training_feature_array, hidden1_weights, hidden1_bias,   
                                                        hidden2_weights, hidden2_bias, out_weights, out_bias,
                                                        target_matrix)
        
        (out_weights, out_bias, hidden2_weights, 
         hidden2_bias, hidden1_weights, hidden1_bias)= gradient_descent_update(learning_rate, out_weights, 
                                                                                    out_bias, out_bias_gradients, 
                                                                                    out_weight_gradients, hidden2_weights, 
                                                                                    hidden2_bias, hidden2_bias_gradients, 
                                                                                    hidden2_weight_gradients, hidden1_weights, 
                                                                                    hidden1_bias, hidden1_bias_gradients, 
                                                                                    hidden1_weight_gradients)
        validation_loss = forward_pass_1HL(validation_features,  hidden1_weights, 
                                            hidden1_bias, hidden2_weights, 
                                            hidden2_bias, out_weights, out_bias,
                                            validation_target)[2]
        
        if validation_loss == previous_validation_loss:
            no_change_epochs += 1 
        else:
            no_change_epochs = 0
        
        previous_validation_loss = validation_loss

        if no_change_epochs >= k:
            stop_descent = True

        epoch_count += 1 
        if epoch_count >= max_epochs:
            stop_descent = True 
            

        
    return out_weights, out_bias, hidden2_weights, hidden2_bias, hidden1_weights, hidden1_bias  
        

## Expanding Activation Function for Activation Class
While I have used the ReLu activation function so far, there are obviously many other options with various use cases, thus to create a proper neural net class, the user must have the option to select their choose of activation function. To ensure clean, readable and maintainable code, for the implementaion of the Neural Net Class, job of running the required activation function given the user input will be handled by a seperate class. This will allow for the easy adding of more activation functions without making an mess of the neuralnet.py file, ensuring modular logic handling. In that vain, it is required that a seperate function be created for each activation function or both the forward and backwards pass.  

## ReLU Function 
Starting with ReLU as it is the one I have the most comfort with given it has formed the bases of my neural net implemenetations so far 

In [ ]:
def relu_forward(z: np.ndarray)-> np.ndarray:
    '''
    Conducts the ReLU activation function on the inputed value for the forward pass   

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 

    Return: 
        np.ndarray: neuron activations
    '''
        relu_z = np.maximum(z, 0)
        return relu_z 

In [ ]:
def relu_backward(z: np.ndarray)-> np.ndarray:
    '''
    Conducts the derivative of ReLU activation function on the inputed value for back propagation

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 

    Return: 
        np.ndarray: deriavtive results 
    '''
        relu_mask = z > 0  
        relu_derivative = relu_mask.astype(int)
        return relu_derivative 

## Sigmoid Function
Now we move to the sigmoid activation function, which takes the form p(x) = 1/1+e^-x and has derivative form p(x)(1-p(X))

In [2]:
def sigmoid_forward(z: np.ndarray)-> np.ndarray:
    '''
    Conducts the Sigmoid activation function on the inputed value for the forward pass   

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 

    Return: 
        np.ndarray: neuron activations
    '''
    sigmoid_z = 1/(1 + np.exp(-1*z))
    return sigmoid_z

In [2]:
def sigmoid_backward(z: np.ndarray)-> np.ndarray:
    '''
    Conducts the derivative of ReLu activation function on the inputed value for back propagation

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 

    Return: 
        np.ndarray: deriavtive results 
    '''
    sigmoid_z = sigmoid_forward(z)
    sigmoid_derivative = sigmoid_z * (1- sigmoid_z)
    return sigmoid_derivative

NameError: name 'np' is not defined

## Leaky ReLU
Leaky ReLU is next with the conditional form of returning z if the activation is above 0 and alpha * z when z is below 0, where alpha is usually a small number like 0.01. Derivative sees the condition being to return a if above 0 and aplha if below. 

In [ ]:
def leaky_relu_forward(z: np.ndarray alpha: float = 0.01)-> np.ndarray:
    '''
    Conducts the Leaky ReLU activation function on the inputed value for the forward pass   

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 
        aplha (float): A hyperparameter that controls the slope for negative inputs 
    Return: 
        np.ndarray: neuron activations
    '''
        leaky_relu_z = np.maximum(z, alpha * z )
        return leaky_relu_z 

In [ ]:
def leaky_relu_backward(z: np.ndarray, alpha: float = 0.01)-> np.ndarray:
    '''
    Conducts the derivative of ReLU activation function on the inputed value for back propagation

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 

    Return: 
        np.ndarray: deriavtive results 
    ''' 
        leaky_relu_derivative = np.where(x > 0, 1, alpha)
        return leaky_relu_derivative

## Hyperbolic Tangent (tanh)
This function has the form tanh(z) = (e ^ z - e ^ -z)/ (e ^ z + e ^ -z) and has the derivative form, 
tanh'(z) = 1 - tanh^2(z)   

In [ ]:
def tanh_forward(z: np.ndarray)-> np.ndarray:
    '''
    Conducts the Tanh activation function on the inputed value for the forward pass   

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 
        aplha (float): A hyperparameter that controls the slope for negative inputs 
    Return: 
        np.ndarray: neuron activations
    '''
    tanh_z = (np.exp(z) - np.exp(-z))/(np.exp(z) + np.exp(-z))
    return tanh_z

In [ ]:
def tanh_backward(z: np.ndarray)-> np.ndarray:
    '''
    Conducts the derivative of ReLu activation function on the inputed value for back propagation

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 

    Return: 
        np.ndarray: deriavtive results 
    ''' 
    tanh_z = (np.exp(z) - np.exp(-z))/(np.exp(z) + np.exp(-z))
    tanh_derivative = 1 - (tanh ** 2 )
    return tanh_derivative

## Softmax
The Softmax function takes the form f(x) = e^z_i/Summation of e^z_i from 1 to K. The derivative takes the form of is a jacobian matrix of the partial derivatives of f(x) with respect to z. 

In [6]:
def softmax_forward(z):
    '''
    Conducts the Softmax activation function on the inputed value for the forward pass   

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 
    Return: 
        np.ndarray: neuron activations
    '''
    e_z = np.exp(z)
    exp_sum = np.sum(e_z)
    softmax_z = e_z/exp_sum
    return softmax_z

In [4]:
def softmax_backwards(z):
    '''
    Conducts the derivative of Softmax activation function on the inputed value for back propagation

    Args: 
        z (np.ndarray): Weighted sum computed through the weights and biases 

    Return: 
        np.ndarray: derivative results 
    ''' 
    softmax_z = softmax_forward(z)
    softmax_derivative = softmax_z.T @ (-1 * softmax_z) + (np.identity(softmax_z.shape[1]) * softmax_z ) # 
    return softmax_derivative

SyntaxError: incomplete input (291444679.py, line 1)